In [1]:
import os
import openai
import duckdb
import pandas as pd
import time
import json

duckdb.sql("""
           CREATE OR REPLACE TABLE component AS
           SELECT "ch.chat_id" AS chat_id
           FROM read_csv('analysis_data/chats_graph/all_components.csv')
           """)

duckdb.sql("""
           CREATE OR REPLACE TABLE chats_all AS
           SELECT chat_id, text_content
           FROM read_csv('preprocess_data/chat_nodes.csv')
           """)

duckdb.sql("""
           CREATE OR REPLACE TABLE chats AS
           SELECT b.chat_id as chat_id, b.text_content as text_content
           FROM component as a, chats_all as b
           WHERE a.chat_id == b.chat_id
           """)

In [2]:
df = duckdb.sql("SELECT * FROM chats").df()

In [3]:
total_chars = int(df['text_content'].str.len().sum())
total_words = int(df['text_content'].str.split().str.len().sum())
print(total_chars, total_words)

346399797 53253791


In [4]:
client = openai.OpenAI(api_key=os.environ.get("MARITACA_API_KEY"),
                       base_url="https://chat.maritaca.ai/api")

prompt_system = """Você é um classificador de desinformação. Analise o texto fornecido e classifique o assunto principal na melhor categoria. Sua resposta deve conter o número de uma das categorias, somente o número.

1. Política e Eleições: Fraude em urnas, ataque a instituições públicas.
2. Economia: Impostos, inflação, crise, PIX, Bolsa Família, financiamento.
3. Saúde e Ciência: Vacinas, tratamentos, alimentação, crise sanitária, pseudociências, ceticismo sobre ciência.
4. Segurança e Golpes: Link de prêmio, alarmismo sobre crimes e golpes.
5. Sociedade: Preconceitos, discriminação de minorias, pessoas e etnias.
6. Calamidades: Invasão, apocalipse, guerra.
7. Indeterminado: apenas se não houver relação com algum assunto anterior.
"""

In [5]:
def batch_jsonl(df, chunk_size=40000, file_prefix="chats_batch"):
    current_file_idx = 0
    written_lines = 0
    file = None

    try:
        for row in df.itertuples(index=False):

            if written_lines == 0:
                filename = f"analysis_data/chats_graph/api_batch/{file_prefix}_{current_file_idx}.jsonl"
                file = open(filename, "w", encoding="utf-8")

            req = {
                "custom_id": str(row.chat_id),
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": "sabia-4",
                    "messages": [
                        {"role": "system", "content": prompt_system},
                        {"role": "user", "content": str(row.text_content)},
                    ],
                    "max_tokens": 5,
                    "tools": [],
                    "temperature": 0.0,
                },
            }

            line = json.dumps(req, ensure_ascii=False)
            file.write(line + "\n")
            written_lines += 1

            if written_lines >= chunk_size:
                file.close()
                file = None
                current_file_idx += 1
                written_lines = 0

    finally:
        if file is not None:
            file.close()

def get_response(string):
    for attempt in range(1,6):
        try:
            response = client.chat.completions.create(
                model="sabia-4",
                messages=[
                    {"role": "system", "content": prompt_system},
                    {"role": "user", "content": string}
                ],
                tools=[],
                max_tokens=5,
                stream=False,
                temperature=0.0
            )
            return response
        except openai.RateLimitError:
            print("HTTP 429, attempt", attempt)
            time.sleep(2*attempt)
    raise Exception

def run_api(classification, df):
    i = 1
    for string in df['text_content']:
        try:
            res = get_response(string)
        except Exception:
            print("Stopping. Failed at index", i)
            return
        classification.append(res.choices[0].message.content)
        time.sleep(1.0)
        i += 1

In [6]:
df = duckdb.sql("SELECT * FROM chats").df()
#classification = []
#run_api(classification, df)
batch_jsonl(df)

In [ ]:
def add_content_classification(df, response_file_path):
    classifications = {}

    with open(response_file_path, "r", encoding="utf-8") as file:
        for line in file:
            data = json.loads(line.strip())

            custom_id = int(data["custom_id"])
            content = data["response"]["body"]["choices"][0]["message"]["content"].strip()
            if content.isdigit() and content in ["1", "2", "3", "4", "5", "6", "7"]:
                classifications[custom_id] = int(content)
            else:
                classifications[custom_id] = 7
    
    df["content_classification"] = df["chat_id"].map(classifications)

    return df